### Adding GPU

In [1]:
##import os
##os.environ["CUDA_VISIBLE_DEVICES"]="XX"

### Loading the necessary libraries

In [ ]:
import os
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from datasets import Dataset
import json
from tqdm import tqdm


### Giving model paths & tokenizers

In [3]:
model_paths = {
    "baseline": "./model_baseline",
    "mrsty": "./model_mrsty",
    "mrsty_carry": "./model_mrsty_carry",
    "mrrel": "./model_mrrel",
    "mrrel_carry": "./model_mrrel_carry",
    "mrrel_fine_1": "./model_mrrel_fine_1",
    "model_mrsty_2": "./model_mrsty_2",
    "model_mrrel_2": "./model_mrrel_2"
}


In [4]:
tokenizer_paths = {
    "baseline": "./tokenizer_baseline",
    "mrsty": "./tokenizer_mrsty",
    "mrsty_carry": "./tokenizer_mrsty_carry",
    "mrrel": "./tokenizer_mrrel",
    "mrrel_carry": "./tokenizer_mrrel_carry",
    "mrrel_fine_1": "./tokenizer_mrrel_carry",
    "model_mrsty_2": "./tokenizer_mrsty_2",
    "model_mrrel_2": "./tokenizer_mrrel_2"
}

### Function to calculate metrics

In [5]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="macro")
    acc = accuracy_score(labels, predictions)
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

In [6]:
def evaluate_model(model_path, tokenizer_path, train_dataset, val_dataset, test_dataset):
    # Load model and tokenizer
    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
    
    # Define Trainer for evaluation
    trainer = Trainer(
        model=model,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics
    )

    # Evaluate on datasets
    print(f"Evaluating model in {model_path}...")
    train_results = trainer.evaluate(train_dataset)
    val_results = trainer.evaluate(val_dataset)
    test_results = trainer.evaluate(test_dataset)

    return {
        "train": train_results,
        "validation": val_results,
        "test": test_results,
    }


### Loading datasets

In [7]:
# Load the baseline tokenizer
baseline_tokenizer = AutoTokenizer.from_pretrained("./tokenizer_baseline")

# Label mapping for the classification task
label_mapping = {"yes": 0, "no": 1, "maybe": 2}

In [8]:
def load_and_process_data(filepath, tokenizer, label_mapping, max_length=512):
    """
    Load and process the preprocessed dataset for a given tokenizer.
    """
    with open(filepath, 'r') as f:
        # Ensure the file is loaded as a dictionary
        data = json.load(f)
    
    # Verify the type of `data`
    if not isinstance(data, dict):
        raise ValueError("The loaded data is not in the expected dictionary format.")
    
    processed_data = {
        "input_ids": [],
        "attention_mask": [],
        "label": []
    }
    
    for pmid, entry in tqdm(data.items(), desc="Processing data"):
        question = entry.get("question", "")
        context = entry.get("context", "")
        label = entry.get("label", None)
        
        if label is None or question == "" or context == "":
            # Skip entries with missing fields
            continue
        
        # Tokenize the question and context using the provided tokenizer
        inputs = tokenizer(
            question + " " + context,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="np"
        )
        
        processed_data["input_ids"].append(inputs["input_ids"][0])
        processed_data["attention_mask"].append(inputs["attention_mask"][0])
        processed_data["label"].append(label_mapping[label])
    
    return Dataset.from_dict(processed_data)

In [9]:
train_dataset_bio = load_and_process_data("preprocessed/processed_train_set.json", baseline_tokenizer, label_mapping)
dev_dataset_bio = load_and_process_data("preprocessed/processed_dev_set.json", baseline_tokenizer, label_mapping)
test_dataset_bio = load_and_process_data("preprocessed/processed_test_set.json", baseline_tokenizer, label_mapping)


Processing data: 100%|███████████████████████| 100/100 [00:00<00:00, 892.70it/s]


In [10]:
def load_from_json(file_path):
   
    with open(file_path, "r") as f:
        data_dict = json.load(f)
    return Dataset.from_dict(data_dict)

In [11]:
# Load MRSTY datasets
train_dataset_mrsty = load_from_json("final_biobert/train_dataset_mrsty.json")
dev_dataset_mrsty = load_from_json("final_biobert/dev_dataset_mrsty.json")
test_dataset_mrsty = load_from_json("final_biobert/test_dataset_mrsty.json")

# Load MRREL datasets
train_dataset_mrrel = load_from_json("final_biobert/train_dataset_mrrel.json")
dev_dataset_mrrel = load_from_json("final_biobert/dev_dataset_mrrel.json")
test_dataset_mrrel = load_from_json("final_biobert/test_dataset_mrrel.json")

print("Datasets loaded successfully")

Datasets loaded successfully


In [12]:
datasets = {
    "baseline": {
        "train": train_dataset_bio,
        "validation": dev_dataset_bio,
        "test": test_dataset_bio,
    },
    "mrsty_carry": {
        "train": train_dataset_mrsty,
        "validation": dev_dataset_mrsty,
        "test": test_dataset_mrsty,
    },
    "mrsty": {
        "train": train_dataset_mrsty,
        "validation": dev_dataset_mrsty,
        "test": test_dataset_mrsty,
    },
    "mrrel_carry": {
        "train": train_dataset_mrrel,
        "validation": dev_dataset_mrrel,
        "test": test_dataset_mrrel,
    },
    "mrrel": {
        "train": train_dataset_mrrel,
        "validation": dev_dataset_mrrel,
        "test": test_dataset_mrrel,
    },
    "mrrel_fine_1": {
        "train": train_dataset_mrrel,
        "validation": dev_dataset_mrrel,
        "test": test_dataset_mrrel,
    },   
    "model_mrsty_2": {
        "train": train_dataset_mrsty,
        "validation": dev_dataset_mrsty,
        "test": test_dataset_mrsty,
    },
    "model_mrrel_2": {
        "train": train_dataset_mrrel,
        "validation": dev_dataset_mrrel,
        "test": test_dataset_mrrel,
    },
}

In [14]:
results_list = []

for model_name, model_path in model_paths.items():
    tokenizer_path = tokenizer_paths[model_name]
    dataset = datasets[model_name]  # Retrieve the datasets for the current model
    
    metrics = evaluate_model(
        model_path, 
        tokenizer_path, 
        dataset["train"], 
        dataset["validation"], 
        dataset["test"]
    )
    
    # Append results for each split to the results list
    for split in ["train", "validation", "test"]:
        results_list.append({
            "Model": model_name,
            "Dataset": split.capitalize(),
            "Accuracy": metrics[split]["eval_accuracy"],
            "Precision": metrics[split]["eval_precision"],
            "Recall": metrics[split]["eval_recall"],
            "F1": metrics[split]["eval_f1"],
        })


/home/stu14/s1/mmm5508/miniconda3/envs/idai610/lib/python3.10/site-packages/transformers/modeling_utils.py:488: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.lo

Evaluating model in ./model_baseline...


/home/stu14/s1/mmm5508/miniconda3/envs/idai610/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1334: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/stu14/s1/mmm5508/miniconda3/envs/idai610/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1334: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/stu14/s1/mmm5508/miniconda3/envs/idai610/lib/python3.10/site-packages/transformers/modeling_utils.py:488: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle d

Evaluating model in ./model_mrsty...


/home/stu14/s1/mmm5508/miniconda3/envs/idai610/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1334: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/stu14/s1/mmm5508/miniconda3/envs/idai610/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1334: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/stu14/s1/mmm5508/miniconda3/envs/idai610/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1334: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

...

Evaluating model in ./model_mrrel_fine_1...


/home/stu14/s1/mmm5508/miniconda3/envs/idai610/lib/python3.10/site-packages/transformers/modeling_utils.py:488: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.lo

Evaluating model in ./model_mrsty_2...


/home/stu14/s1/mmm5508/miniconda3/envs/idai610/lib/python3.10/site-packages/transformers/modeling_utils.py:488: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.lo

Evaluating model in ./model_mrrel_2...


In [16]:
# Create DataFrame
results_df = pd.DataFrame(results_list)

# Print results as a table
results_df


,Model,Dataset,Accuracy,Precision,Recall,F1
0,baseline,Train,0.90750,0.931814,0.727374,0.738931
1,baseline,Validation,0.61000,0.403846,0.491830,0.439482
2,baseline,Test,0.64000,0.421537,0.461589,0.439996
3,mrsty,Train,0.87750,0.596154,0.655475,0.623909
4,mrsty,Validation,0.87750,0.596154,0.655475,0.623909
5,mrsty,Test,0.59000,0.390339,0.428337,0.406526
6,mrsty_carry,Train,0.99625,0.996794,0.988235,0.992404
7,mrsty_carry,Validation,0.99625,0.996794,0.988235,0.992404
8,mrsty_carry,Test,0.57000,0.440741,0.440029,0.439869
9,mrrel,Train,0.87750,0.932158,0.658381,0.637806
